In [4]:
import numpy as np
import pandas as pd

def generate_realistic_upi_dataset(n_rows=100000, fraud_rate=0.015, seed=42):
    rng = np.random.default_rng(seed)

    start_date = pd.Timestamp("2025-01-01")
    day_offsets = np.sort(rng.integers(0, 120, size=n_rows))
    dates = (start_date + pd.to_timedelta(day_offsets, unit="D")).strftime("%d/%m/%y")

    fraud = (rng.random(n_rows) < fraud_rate).astype(int)
    is_late_period = day_offsets >= 96  # Last 20% test period experiences temporal drift

    # Overlapping monetary & behavioral distributions (no trivial threshold separation)
    base_amount = rng.lognormal(mean=6.3, sigma=1.25, size=n_rows)
    fraud_boost = np.where(
        fraud == 1,
        np.where(is_late_period, rng.lognormal(0.55, 0.5, n_rows), rng.lognormal(0.95, 0.5, n_rows)),
        1.0
    )
    amount = np.round(np.clip(base_amount * fraud_boost, 10.0, 100000.0), 2)

    # Heavy overlap in deviation, frequency, and recency
    deviation = np.round(
        np.where(fraud == 1, rng.normal(140.0, 160.0, n_rows), rng.normal(10.0, 95.0, n_rows)), 2
    )
    frequency = np.where(
        fraud == 1,
        np.where(is_late_period, rng.poisson(3.2, n_rows), rng.poisson(4.5, n_rows)),
        rng.poisson(2.1, n_rows)
    )
    days_since = np.where(
        fraud == 1, rng.exponential(4.5, n_rows).astype(int), rng.exponential(11.0, n_rows).astype(int)
    )

    states = ["Maharashtra", "Karnataka", "Delhi", "Tamil Nadu", "Uttar Pradesh", "Gujarat", "Telangana", "West Bengal"]
    channels = ["Mobile", "Online", "In-store"]
    os_list = ["Android", "iOS", "Windows", "MacOS"]
    statuses = ["Completed", "Pending", "Failed"]

    tx_state = rng.choice(states, size=n_rows, p=[0.22, 0.18, 0.15, 0.12, 0.11, 0.08, 0.08, 0.06])
    tx_channel = np.where(
        fraud == 1,
        rng.choice(channels, size=n_rows, p=[0.48, 0.44, 0.08]),
        rng.choice(channels, size=n_rows, p=[0.62, 0.23, 0.15])
    )
    device_os = np.where(
        fraud == 1,
        rng.choice(os_list, size=n_rows, p=[0.62, 0.16, 0.18, 0.04]),
        rng.choice(os_list, size=n_rows, p=[0.70, 0.22, 0.05, 0.03])
    )
    tx_status = np.where(
        fraud == 1,
        rng.choice(statuses, size=n_rows, p=[0.78, 0.15, 0.07]),
        rng.choice(statuses, size=n_rows, p=[0.90, 0.06, 0.04])
    )

    # Shared CGNAT / Public Wi-Fi IP pool (realistic overlap between clean and fraud traffic)
    shared_ip_pool = [f"103.{rng.integers(10,120)}.{rng.integers(1,120)}.{rng.integers(1,120)}" for _ in range(18000)]
    suspicious_subset = shared_ip_pool[:350]  # Fraudsters reuse a subset of shared carrier IPs

    ip_address = []
    for f_flag, late in zip(fraud, is_late_period):
        if f_flag == 1 and (not late) and rng.random() < 0.65:
            ip_address.append(rng.choice(suspicious_subset))
        elif f_flag == 1 and late and rng.random() < 0.35:  # Fraudsters rotate IPs in test window
            ip_address.append(rng.choice(suspicious_subset))
        else:
            ip_address.append(rng.choice(shared_ip_pool))

    df = pd.DataFrame({
        "Transaction_ID": [f"TXN{1000000 + i}" for i in range(n_rows)],
        "Date": dates,
        "amount": amount,
        "Transaction_Amount_Deviation": deviation,
        "Transaction_Frequency": frequency,
        "Days_Since_Last_Transaction": days_since,
        "Transaction_Status": tx_status,
        "Device_OS": device_os,
        "Transaction_Channel": tx_channel,
        "Transaction_State": tx_state,
        "IP_Address": ip_address,
        "fraud": fraud
    })

    df.to_csv("UPI_FRAUD_100K.csv", index=False)
    print(f"Generated realistic UPI_FRAUD_100K.csv ({len(df):,} rows, {df['fraud'].sum():,} fraud cases).")

generate_realistic_upi_dataset()


Generated realistic UPI_FRAUD_100K.csv (100,000 rows, 1,563 fraud cases).


In [5]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.ensemble import IsolationForest
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import average_precision_score, roc_auc_score, roc_curve
import xgboost as xgb
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# 1. Load & Chronologically Sort Data (No Future Leakage)
# ---------------------------------------------------------
def load_and_split_temporal(file_path="UPI_FRAUD_100K.csv", train_ratio=0.8):
    df = pd.read_csv(file_path)
    df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%y", errors="coerce")
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
    df["Transaction_Amount_Deviation"] = pd.to_numeric(df["Transaction_Amount_Deviation"], errors="coerce")
    df["fraud"] = df["fraud"].astype(int)
    df = df.dropna(subset=["Date", "amount", "fraud"]).sort_values("Date").reset_index(drop=True)

    split_idx = int(len(df) * train_ratio)
    train_df = df.iloc[:split_idx].copy()
    test_df = df.iloc[split_idx:].copy()
    print(f"Dataset loaded: {len(df):,} rows | Fraud rate: {df['fraud'].mean()*100:.2f}%")
    print(f"Temporal Split -> Train: {len(train_df):,} | Test: {len(test_df):,}")
    return train_df, test_df

# ---------------------------------------------------------
# 2. Leakage-Free Relational & Behavioral Feature Engineering
# ---------------------------------------------------------
def engineer_features(train_df, test_df):
    # Compute IP-level relational risk features strictly from training window
    if "IP_Address" in train_df.columns:
        ip_counts = train_df["IP_Address"].value_counts()
        ip_state_diversity = train_df.groupby("IP_Address")["Transaction_State"].nunique()

        for split in (train_df, test_df):
            split["ip_txn_count"] = split["IP_Address"].map(ip_counts).fillna(0)
            split["ip_state_diversity"] = split["IP_Address"].map(ip_state_diversity).fillna(1)
    else:
        for split in (train_df, test_df):
            split["ip_txn_count"] = 0
            split["ip_state_diversity"] = 1

    num_cols = [
        "amount", "Transaction_Amount_Deviation",
        "Transaction_Frequency", "Days_Since_Last_Transaction",
        "ip_txn_count", "ip_state_diversity"
    ]
    cat_cols = [c for c in ["Transaction_Status", "Device_OS", "Transaction_Channel", "Transaction_State"] if c in train_df.columns]

    encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    train_cat = encoder.fit_transform(train_df[cat_cols].astype(str))
    test_cat = encoder.transform(test_df[cat_cols].astype(str))

    scaler = StandardScaler()
    train_num = scaler.fit_transform(train_df[num_cols].fillna(0))
    test_num = scaler.transform(test_df[num_cols].fillna(0))

    X_train = np.hstack([train_num, train_cat]).astype(np.float32)
    X_test = np.hstack([test_num, test_cat]).astype(np.float32)
    y_train = train_df["fraud"].values
    y_test = test_df["fraud"].values
    return X_train, y_train, X_test, y_test

# ---------------------------------------------------------
# 3. PyTorch Focal-Loss Tabular MLP
# ---------------------------------------------------------
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        probs = torch.sigmoid(logits)
        pt = targets * probs + (1 - targets) * (1 - probs)
        alpha_t = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        loss = alpha_t * ((1 - pt) ** self.gamma) * bce
        return loss.mean()

class FraudMLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64),
            nn.BatchNorm1d(64),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.SiLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

def train_pytorch_focal_mlp(X_train, y_train, X_test, epochs=12, batch_size=512):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = FraudMLP(X_train.shape[1]).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = FocalLoss(alpha=0.75, gamma=2.0)

    train_loader = DataLoader(
        TensorDataset(torch.tensor(X_train), torch.tensor(y_train, dtype=torch.float32)),
        batch_size=batch_size, shuffle=True
    )

    model.train()
    for _ in range(epochs):
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(X_test).to(device))
        probs = torch.sigmoid(logits).cpu().numpy()
    return probs

# ---------------------------------------------------------
# 4. Operational Imbalance & Calibration Evaluation
# ---------------------------------------------------------
def compute_metrics(y_true, y_prob, model_name, fpr_target=0.01, n_bins=10):
    pr_auc = average_precision_score(y_true, y_prob)
    roc_auc = roc_auc_score(y_true, y_prob)

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    recall_1fpr = tpr[np.searchsorted(fpr, fpr_target, side="right") - 1]

    # Expected Calibration Error (ECE)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (y_prob >= bins[i]) & (y_prob < bins[i + 1])
        if np.any(mask):
            ece += (np.sum(mask) / len(y_true)) * abs(np.mean(y_true[mask]) - np.mean(y_prob[mask]))

    return {
        "Model": model_name,
        "PR-AUC": round(float(pr_auc), 4),
        "ROC-AUC": round(float(roc_auc), 4),
        "Recall@1%FPR": round(float(recall_1fpr), 4),
        "ECE": round(float(ece), 4)
    }

# ---------------------------------------------------------
# 5. Run Complete Benchmark
# ---------------------------------------------------------
if __name__ == "__main__":
    # Update path if running on Colab vs local Windows machine
    csv_path = "UPI_FRAUD_100K.csv" if os.path.exists("UPI_FRAUD_100K.csv") else r"C:\Users\samik\OneDrive\Desktop\IP(3RD SEM)\UPI_FRAUD_100K.csv"
    train_df, test_df = load_and_split_temporal(csv_path)
    X_train, y_train, X_test, y_test = engineer_features(train_df, test_df)

    results = []

    # Baseline 1: Unsupervised Isolation Forest (fixed continuous anomaly score)
    iso = IsolationForest(contamination=float(y_train.mean()), random_state=42, n_jobs=-1)
    iso.fit(X_train)
    raw_scores = -iso.decision_function(X_test)
    iso_probs = (raw_scores - raw_scores.min()) / (raw_scores.max() - raw_scores.min() + 1e-8)
    results.append(compute_metrics(y_test, iso_probs, "Isolation Forest (Unsupervised)"))

    # Model 2: Cost-Sensitive XGBoost
    pos_weight = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)
    xgb_clf = xgb.XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.05,
        scale_pos_weight=pos_weight, eval_metric="aucpr", random_state=42, n_jobs=-1
    )
    xgb_clf.fit(X_train, y_train)
    xgb_probs = xgb_clf.predict_proba(X_test)[:, 1]
    results.append(compute_metrics(y_test, xgb_probs, "XGBoost (Cost-Weighted)"))

    # Model 3: Isotonic-Calibrated XGBoost
    cal_xgb = CalibratedClassifierCV(xgb_clf, method="isotonic", cv=3)
    cal_xgb.fit(X_train, y_train)
    cal_probs = cal_xgb.predict_proba(X_test)[:, 1]
    results.append(compute_metrics(y_test, cal_probs, "XGBoost + Isotonic Calibration"))

    # Model 4: PyTorch Focal-Loss MLP
    mlp_probs = train_pytorch_focal_mlp(X_train, y_train, X_test, epochs=12)
    results.append(compute_metrics(y_test, mlp_probs, "PyTorch Focal-Loss MLP"))

    summary = pd.DataFrame(results)
    summary.to_csv("upi_research_benchmark.csv", index=False)
    print("\n=== Temporal UPI Fraud Benchmark Results ===")
    print(summary.to_string(index=False))


Dataset loaded: 100,000 rows | Fraud rate: 1.56%
Temporal Split -> Train: 80,000 | Test: 20,000

=== Temporal UPI Fraud Benchmark Results ===
                          Model  PR-AUC  ROC-AUC  Recall@1%FPR    ECE
Isolation Forest (Unsupervised)  0.0332   0.6909        0.0185 0.2879
        XGBoost (Cost-Weighted)  0.3268   0.8790        0.3426 0.1154
 XGBoost + Isotonic Calibration  0.3322   0.8829        0.3488 0.0049
         PyTorch Focal-Loss MLP  0.3741   0.8969        0.3889 0.1176
